# Multi-Agent System - 使用 CrewAI 框架

## 目標

- 理解 CrewAI 的核心概念（Agent, Task, Crew）
- 掌握不同協作模式：Sequential, Parallel, Hierarchical
- 設計並實作 Multi-Agent System

---

## 環境設定

### 安裝必要套件

In [ ]:
# 安裝 CrewAI
!pip install -q --upgrade crewai

### 設定 API 金鑰

您需要一個 Gemini API 金鑰才能使用本實驗。

**取得 API 金鑰的步驟：**

1. 前往 [Google AI Studio](https://aistudio.google.com/app/apikey)
2. 建立一個新的 API 金鑰
3. 在 Colab 左側選單中，點擊 🔑 圖示（Secrets）
4. 新增一個名為 `GOOGLE_API_KEY` 的 secret
5. 貼上您的 API 金鑰並儲存
6. 啟用該 secret 的存取權限

In [ ]:
import os
from google.colab import userdata

# 從 Colab Secrets 取得 API 金鑰
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ API 金鑰設定完成")
except Exception as e:
    print(f"❌ 錯誤：請確認您已在 Colab Secrets 中新增 'GOOGLE_API_KEY'")
    print(f"詳細資訊：{e}")

### 驗證設置

In [ ]:
from crewai import Agent, Task, Crew, LLM

# 建立 Agent，使用 Gemini 作為大腦
test_agent = Agent(
    role='Tester',
    goal='驗證系統設置是否正確',
    backstory='你是負責驗證系統的測試專員',
    llm=LLM(model="gemini/gemini-flash-latest")
)

# 建立 Task
test_task = Task(
    description='請回答：CrewAI 設置成功了嗎？',
    agent=test_agent,
    expected_output='簡短的確認訊息'
)

# 建立團隊
test_crew = Crew(
    name='Test Crew',
    agents=[test_agent],
    tasks=[test_task],
    verbose=True
)

result = test_crew.kickoff()
print("\n✅ 環境設置驗證完成！")
print(f"測試結果: {result}")

## Part 1: Sequential 模式 - Two Agents

情境：新聞蒐集 → 摘要撰寫

### Step 1.1: 定義 Agents

In [ ]:
# 新聞蒐集員
news_collector = Agent(
    role='新聞蒐集員',
    goal='蒐集最新且重要的新聞',
    backstory='你是資深記者，擅長發掘重要新聞',
    llm=LLM(model="gemini/gemini-flash-latest")
)

# 摘要撰寫員
summarizer = Agent(
    role='摘要撰寫員',
    goal='將新聞濃縮成簡短易懂的摘要',
    backstory='你擅長精簡資訊，用最少文字傳達重點',
    llm=LLM(model="gemini/gemini-flash-latest")
)

print("✅ Agents 建立完成")

### Step 1.2: 定義任務（Sequential）

In [ ]:
# 設定參數
news_topic = "人工智慧"
news_count = 3

# 任務 1：蒐集新聞
collect_task = Task(
    description=f"""蒐集關於「{news_topic}」的最新新聞：

    請提供：
    1. {news_count} 則重要新聞標題
    2. 每則新聞的來源
    3. 發布日期
    4. 簡短描述（1-2 句話）

    請確保新聞的時效性和可信度。
    """,
    agent=news_collector,
    expected_output="新聞標題清單與簡短描述"
)

# 任務 2：撰寫摘要
summarize_task = Task(
    description=f"""根據蒐集到的新聞，撰寫一份整合摘要：

    請提供：
    1. 一段式整合摘要（150 字內）
    2. 關鍵趨勢或重點
    3. 對讀者的意義

    請用淺顯易懂的文字，避免過多專業術語。
    """,
    agent=summarizer,
    expected_output="整合式新聞摘要",
    context=[collect_task]  # 依賴蒐集任務的結果
)

print("✅ Sequential 任務定義完成")

### Step 1.3: 執行 Sequential 流程

In [ ]:
# 建立 Sequential Crew
news_crew = Crew(
    name='新聞摘要團隊',
    agents=[news_collector, summarizer],
    tasks=[collect_task, summarize_task],
    process="sequential",
    verbose=True
)

print("="*50)
print(f"🚀 開始蒐集「{news_topic}」相關新聞")
print("="*50)

result = news_crew.kickoff()

print("\n" + "="*50)
print("✅ 新聞摘要完成！")
print("="*50)
print(f"\n{result}")

---

## Part 2: Sequential 模式 - Three Agents

加入新聞評論員

### Step 2.1: 定義 Agent

In [ ]:
# 新聞評論員
commentator = Agent(
    role='新聞評論員',
    goal='提供深度分析和專業見解',
    backstory='你是資深產業分析師，擅長解讀新聞背後的意義和影響',
    llm=LLM(model="gemini/gemini-flash-latest")
)

print("✅ 新聞評論 Agent 建立完成")

### Step 2.2: 定義任務

In [ ]:
# 任務 3：評論分析
comment_task = Task(
    description=f"""根據新聞和摘要，提供專業評論：

    請提供：
    1. 深度分析（這些新聞反映什麼趨勢？）
    2. 影響評估（對產業/社會的影響）
    3. 未來展望（接下來可能的發展）
    4. 個人觀點（你的專業見解）

    請以專業但不艱澀的方式呈現。
    """,
    agent=commentator,
    expected_output="專業評論與分析",
    context=[collect_task, summarize_task]  # 可參考前兩個任務的結果
)

print("✅ 評論任務定義完成")

### Step 2.3: 執行 Sequential 流程

In [ ]:
# 建立 Sequential Crew
news_crew = Crew(
    name='新聞摘要與評論團隊',
    agents=[news_collector, summarizer, commentator],
    tasks=[collect_task, summarize_task, comment_task],
    process="sequential",  # 順序執行：蒐集 → 摘要 → 評論
    verbose=True
)

print("="*50)
print(f"🚀 開始製作「{news_topic}」新聞專題")
print("="*50)

result = news_crew.kickoff()

print("\n" + "="*50)
print("✅ 新聞專題完成！")
print("="*50)
print(f"\n{result}")

---

## Part 3: Parallel 模式 - 多領域分析

情境：同時分析旅遊景點與美食地圖

### Step 3.1: 定義三個專家 Agents

In [ ]:
# 景點研究員
attraction_researcher = Agent(
    role='景點研究員',
    goal='找出最值得去的景點',
    backstory='你熟悉各地旅遊景點',
    llm=LLM(model="gemini/gemini-flash-latest")
)

# 美食專家
food_expert = Agent(
    role='美食專家',
    goal='推薦當地必吃美食',
    backstory='你是美食評論家',
    llm=LLM(model="gemini/gemini-flash-latest")
)

# 行程規劃師
trip_planner = Agent(
    role='行程規劃師',
    goal='整合景點和美食，安排完整行程',
    backstory='你擅長規劃旅遊路線',
    llm=LLM(model="gemini/gemini-flash-latest")
)

### Step 3.2: 定義任務

In [ ]:
# 目的地變數
destination = "台北"
duration = "一日遊"

# 景點任務
attraction_task = Task(
    description=f"""針對「{destination} {duration}」，推薦必去景點：

    請提供：
    1. 推薦 3 個必去景點
    2. 每個景點的特色說明
    3. 建議停留時間
    4. 最佳參觀時段
    """,
    agent=attraction_researcher,
    async_execution=True,
    expected_output="景點推薦清單"
)

# 美食任務
food_task = Task(
    description=f"""針對「{destination} {duration}」，推薦必吃美食：

    請提供：
    1. 推薦 3 家特色餐廳或小吃
    2. 每家的招牌菜色
    3. 價格區間
    4. 營業時間
    """,
    agent=food_expert,
    async_execution=True,
    expected_output="美食推薦清單"
)

In [ ]:
# 統籌任務（等待前兩個完成）
coordination_task = Task(
    description=f"""針對「{destination} {duration}」，規劃完整行程：

    請整合景點和美食資訊，提供：
    1. 完整時間表（含早中晚餐安排）
    2. 景點間的移動方式和時間
    3. 預算估算
    4. 注意事項和小提醒

    請確保行程緊湊但不會太趕。
    """,
    agent=trip_planner,
    expected_output="完整行程表",
    context=[attraction_task, food_task]  # 依賴前兩個任務的結果
)

### Step 3.3: 團隊執行

In [ ]:
# 建立 Crew
tech_crew = Crew(
    name='旅遊規劃團隊',
    agents=[attraction_researcher, food_expert, trip_planner],
    tasks=[attraction_task, food_task, coordination_task],
    verbose=True
)

final_result = tech_crew.kickoff()

print("\n" + "="*50)
print("✅ 行程規劃完成！")
print("="*50)
print(final_result)

---

## Part 4: Hierarchical 模式 - 分層決策

由 Manager 協調 Agents，Manager 決定執行順序和任務分配

### Step 4.1: 情境: Party Time!
把 Hierarchical 想像成派對主辦人（Manager），要協調來幫忙的朋友們（Agents）

In [ ]:
# 朋友 1：負責買東西
shopper = Agent(
    role='採購員',
    goal='買派對需要的東西',
    backstory='你很會買東西',
    llm=LLM(model="gemini/gemini-flash-latest")
)

# 朋友 2：負責做菜
chef = Agent(
    role='廚師',
    goal='準備派對食物',
    backstory='你很會做菜',
    llm=LLM(model="gemini/gemini-flash-latest")
)

# 朋友 3：負責布置
decorator = Agent(
    role='布置員',
    goal='布置派對場地',
    backstory='你很會布置',
    llm=LLM(model="gemini/gemini-flash-latest")
)

### Step 4.2: 三件要做的事

In [ ]:
# Tasks
buy_task = Task(
    description='列出派對要買的東西清單（食材、飲料、裝飾品）',
    agent=shopper,
    expected_output='購物清單'
)

cook_task = Task(
    description='決定要做什麼菜（至少 3 道）',
    agent=chef,
    expected_output='菜單'
)

decorate_task = Task(
    description='規劃怎麼布置場地（氣球、燈光、桌椅）',
    agent=decorator,
    expected_output='布置計畫'
)

### Step 4.3: 由 Manager 協調執行

In [ ]:
from crewai import Process

# 使用 Hierarchical Process（Manager 會自動協調迭代）
crew = Crew(
    name='派對籌備團隊',
    agents=[shopper, chef, decorator],
    tasks=[buy_task, cook_task, decorate_task],
    process=Process.hierarchical,  # 👈 Hierarchical 模式！
    manager_llm=LLM(model="gemini/gemini-flash-latest"),  # 👈 Magager 也需要有大腦！
    verbose=True
)

print("🎉 開始籌備派對！")
result = crew.kickoff()
print("✅ 派對準備完成！")

**💡 觀察**  
你的 Manager 有越俎代庖嗎?

在 Hierarchical 模式下，Manager 有時候會自己完成任務而不是分配給對應的 Agent。  
這是因為：
* Manager 覺得「我自己做比較快」
* Task 描述不夠強調「必須由特定 agent 執行」
* Hierarchical 模式本身就比較難預測

Hierarchical **更智能但較難預測**

---

## 總結

**不同協作模式**：
1. **Sequential**: 固定順序（需求 → 開發 → 測試）
2. **Parallel**: 並行執行（多領域同時分析）
4. **Hierarchical**: 階層式協調（經理動態分配）

**模式選擇**：
```
固定順序？→ Sequential
任務獨立？→ Parallel
動態分配？→ Hierarchical
```

**關鍵洞察**：
* **專業分工**：每個 Agent 專注於特定領域
* **模式組合**：複雜系統可以組合多種模式
* **實用框架**：CrewAI 提供業界標準的開發方式